# RQ1 — Content differences across formats

Reproduces the subsection *Differences across formats* of the paper (labels of run 1):

| Result in the paper | Where |
|---|---|
| With all 1,828 files, `AGENTS.md` looks more operational than `CLAUDE.md` | section 1 |
| Subset of repositories with a single file (761) and number of files per format | section 2 |
| Format × category table, Cramér's V, and significant pairs | section 3 |

Method: for each category, a chi-square test of independence on the presence × format table, with
Cramér's V as effect size; when p < 0.05, Fisher's exact test between pairs of formats with Holm's
correction within the category. `GEMINI.md` and `.cursorrules` have fewer than 30 files in the subset
and are left out of the tests.

In [1]:
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figures import save, svg_format_category

RUN_1 = ROOT / "data/rq01/classifier/rq01_classification_run_1.csv"
OUT = ROOT / "results/rq01"
OUT.mkdir(parents=True, exist_ok=True)

# Order of the prevalence table (02_content.ipynb)
CATEGORIES = [
    "test_instructions", "code_conventions", "run_and_build", "ai_rules", "project_overview",
    "doc_metadata", "project_structure", "contribution_guidelines", "system_design",
    "troubleshooting", "security", "deployment_&_infrastructure", "performance_&_resource_management",
]
TESTED_FORMATS = ["AGENTS.md", "CLAUDE.md", "COPILOT"]
ALPHA = 0.05

df = pd.read_csv(RUN_1)
for c in CATEGORIES:
    df[c] = df[c].fillna(0).astype(int)
df["files_in_repo"] = df["repo"].map(df.groupby("repo").size())


def analyze(frame, formats):
    frame = frame[frame["arquivo_analisado"].isin(formats)]
    n_fmt = frame["arquivo_analisado"].value_counts().reindex(formats)
    summary, pairs = [], []
    for cat in CATEGORIES:
        k = frame.groupby("arquivo_analisado")[cat].sum().reindex(formats)
        table = np.array([k.values, (n_fmt - k).values])
        chi2, p, _, _ = chi2_contingency(table)
        v = np.sqrt(chi2 / table.sum())  # 2 × k table: min(rows, columns) - 1 = 1
        row = {"category": cat, **{f"pct_{f}": 100 * k[f] / n_fmt[f] for f in formats},
               "chi2": chi2, "p": p, "cramer_v": v}
        n_sig = 0
        if p < ALPHA:
            combos = list(combinations(formats, 2))
            p_pairs = [fisher_exact([[k[a], k[b]], [n_fmt[a] - k[a], n_fmt[b] - k[b]]])[1] for a, b in combos]
            reject, p_holm, _, _ = multipletests(p_pairs, method="holm")
            for (a, b), pr, ph, rj in zip(combos, p_pairs, p_holm, reject):
                pairs.append({"category": cat, "pair": f"{a} vs {b}", "p_fisher": pr, "p_holm": ph,
                              "significant": bool(rj)})
                n_sig += int(rj)
        row["significant_pairs"] = n_sig
        summary.append(row)
    return pd.DataFrame(summary), pd.DataFrame(pairs), n_fmt

## 1. All files

In [2]:
all_files, _, n_all = analyze(df, TESTED_FORMATS)
print("Files per format (all):", df["arquivo_analisado"].value_counts().to_dict())
all_files.set_index("category").loc[["test_instructions", "run_and_build"],
                                    [f"pct_{f}" for f in TESTED_FORMATS]].round(1)

Files per format (all): {'AGENTS.md': 703, 'CLAUDE.md': 649, 'COPILOT': 409, 'GEMINI.md': 37, 'CURSOR': 30}


,pct_AGENTS.md,pct_CLAUDE.md,pct_COPILOT
category,,,
test_instructions,80.7,61.3,63.3
run_and_build,76.2,61.8,53.3


## 2. Repositories with a single file

In [3]:
single = df[df["files_in_repo"] == 1]
print(f"Repositories with a single downloaded file: {len(single)}")
print(single["arquivo_analisado"].value_counts().to_dict())

Repositories with a single downloaded file: 761
{'AGENTS.md': 273, 'CLAUDE.md': 244, 'COPILOT': 229, 'GEMINI.md': 12, 'CURSOR': 3}


## 3. Format × category table

In [4]:
summary, pairs, n_single = analyze(single, TESTED_FORMATS)
summary.to_csv(OUT / "format_category_single_file.csv", index=False)
pairs.to_csv(OUT / "format_category_single_file_pairs.csv", index=False)

counts = single["arquivo_analisado"].value_counts()
NAME = {"CURSOR": ".cursorrules"}
omitted = " and ".join(f"{NAME.get(f, f)} ({counts[f]})" for f in ["GEMINI.md", "CURSOR"])
labels = {"AGENTS.md": "AGENTS.md", "CLAUDE.md": "CLAUDE.md", "COPILOT": "copilot-instr."}
save(svg_format_category(summary, n_single, TESTED_FORMATS, labels, omitted), OUT / "format-category-single-file.svg")

without_sd = summary[summary["category"] != "system_design"]["cramer_v"]
print(f"Cramér's V without system_design: {without_sd.min():.2f} to {without_sd.max():.2f}")
print(f"V of system_design: {summary.set_index('category').loc['system_design', 'cramer_v']:.2f}")
summary.round(3)

saved /home/keven/Documents/tcc/repo-pesquisa-repro/results/rq01/format-category-single-file.svg
Cramér's V without system_design: 0.03 to 0.21
V of system_design: 0.32


,category,pct_AGENTS.md,pct_CLAUDE.md,pct_COPILOT,chi2,p,cramer_v,significant_pairs
0,test_instructions,79.853,80.738,67.249,14.926,0.001,0.141,2
1,code_conventions,75.458,73.361,72.926,0.491,0.782,0.026,0
2,run_and_build,73.260,80.738,57.642,31.700,0.000,0.206,3
3,ai_rules,62.271,42.213,55.895,21.462,0.000,0.170,2
4,project_overview,54.579,70.082,58.079,13.954,0.001,0.137,2
5,doc_metadata,44.322,34.836,34.934,6.524,0.038,0.094,0
6,project_structure,54.579,59.426,50.655,3.697,0.157,0.070,0
7,contribution_guidelines,58.974,42.623,42.358,18.823,0.000,0.159,2
8,system_design,33.333,65.574,30.568,75.565,0.000,0.318,2
9,troubleshooting,15.018,17.623,24.017,6.896,0.032,0.096,1


In [5]:
pairs[pairs["significant"]].round(4)

,category,pair,p_fisher,p_holm,significant
1,test_instructions,AGENTS.md vs COPILOT,0.0015,0.0032,True
2,test_instructions,CLAUDE.md vs COPILOT,0.0011,0.0032,True
3,run_and_build,AGENTS.md vs CLAUDE.md,0.0478,0.0478,True
4,run_and_build,AGENTS.md vs COPILOT,0.0003,0.0006,True
5,run_and_build,CLAUDE.md vs COPILOT,0.0000,0.0000,True
6,ai_rules,AGENTS.md vs CLAUDE.md,0.0000,0.0000,True
8,ai_rules,CLAUDE.md vs COPILOT,0.0032,0.0065,True
9,project_overview,AGENTS.md vs CLAUDE.md,0.0004,0.0012,True
11,project_overview,CLAUDE.md vs COPILOT,0.0072,0.0144,True
15,contribution_guidelines,AGENTS.md vs CLAUDE.md,0.0002,0.0007,True
